In [9]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import LeaveOneOut
from scipy.stats import wilcoxon

df_m = pd.read_csv('../data/features_lagged.csv')
y = df_m['area_km2']

feature_sets = {
    'Baseline (N + SST)':           ['nitrogen_load', 'sst_c'],
    'Baseline + ENSO':              ['nitrogen_load', 'sst_c', 'oni_spring', 'oni_summer'],
    'Baseline + Lag2':              ['nitrogen_load', 'sst_c', 'nitrogen_lag2'],
    'Baseline + Cumul':             ['nitrogen_load', 'sst_c', 'nitrogen_cumul2'],
    'Full model (all features)':    ['nitrogen_load', 'sst_c', 'oni_spring', 'oni_summer',
                                     'nitrogen_lag2', 'nitrogen_cumul2', 'enso_x_nitrogen'],
}

In [3]:
loo_predictions = {}

for name, features in feature_sets.items():
    X = df_m[features]
    loo = LeaveOneOut()
    preds = []
    for train_idx, test_idx in loo.split(X):
        m = RandomForestRegressor(n_estimators=100, random_state=42)
        m.fit(X.iloc[train_idx], y.iloc[train_idx])
        preds.append(m.predict(X.iloc[test_idx])[0])
    loo_predictions[name] = np.array(preds)

print("Done — predictions stored for:", list(loo_predictions.keys()))

Done — predictions stored for: ['Baseline (N + SST)', 'Baseline + ENSO', 'Baseline + Lag2', 'Baseline + Cumul', 'Full model (all features)']


In [4]:
y_true = y.values
baseline_abs_err = np.abs(y_true - loo_predictions['Baseline (N + SST)'])

for name in feature_sets:
    if name == 'Baseline (N + SST)':
        continue
    alt_abs_err = np.abs(y_true - loo_predictions[name])
    diff = baseline_abs_err - alt_abs_err
    stat, p = wilcoxon(diff)
    print(f"{name:<30} W={stat:.1f}  p={p:.4f}")

Baseline + ENSO                W=333.0  p=0.5955
Baseline + Lag2                W=337.0  p=0.6360
Baseline + Cumul               W=320.0  p=0.4727
Full model (all features)      W=324.0  p=0.5090


In [5]:
rng = np.random.default_rng(0)
n_boot = 5000
n = len(y_true)
baseline_preds = loo_predictions['Baseline (N + SST)']

from sklearn.metrics import r2_score

for name in feature_sets:
    if name == 'Baseline (N + SST)':
        continue
    alt_preds = loo_predictions[name]
    deltas = np.zeros(n_boot)
    for b in range(n_boot):
        idx = rng.choice(n, size=n, replace=True)
        deltas[b] = r2_score(y_true[idx], alt_preds[idx]) - r2_score(y_true[idx], baseline_preds[idx])
    lo, hi = np.percentile(deltas, [2.5, 97.5])
    point = r2_score(y_true, alt_preds) - r2_score(y_true, baseline_preds)
    print(f"{name:<30} ΔR²={point:+.3f}  95% CI=[{lo:+.3f}, {hi:+.3f}]")

Baseline + ENSO                ΔR²=-0.018  95% CI=[-0.123, +0.090]
Baseline + Lag2                ΔR²=-0.014  95% CI=[-0.077, +0.046]
Baseline + Cumul               ΔR²=+0.007  95% CI=[-0.099, +0.133]
Full model (all features)      ΔR²=-0.037  95% CI=[-0.157, +0.076]


In [10]:
seeds = list(range(30))
seed_results = {name: {'r2': [], 'mae': []} for name in feature_sets}

for name, features in feature_sets.items():
    X = df_m[features]
    for s in seeds:
        loo = LeaveOneOut()
        preds = []
        for train_idx, test_idx in loo.split(X):
            m = RandomForestRegressor(n_estimators=100, random_state=s)
            m.fit(X.iloc[train_idx], y.iloc[train_idx])
            preds.append(m.predict(X.iloc[test_idx])[0])
        preds = np.array(preds)
        seed_results[name]['r2'].append(r2_score(y_true, preds))
        seed_results[name]['mae'].append(mean_absolute_error(y_true, preds))

print(f"{'Model':<30}{'R2 mean':>10}{'R2 std':>10}{'MAE mean':>12}{'MAE std':>10}")
for name, r in seed_results.items():
    print(f"{name:<30}{np.mean(r['r2']):>10.3f}{np.std(r['r2']):>10.4f}{np.mean(r['mae']):>12.0f}{np.std(r['mae']):>10.1f}")

Model                            R2 mean    R2 std    MAE mean   MAE std
Baseline (N + SST)                 0.490    0.0124        2651      43.4
Baseline + ENSO                    0.480    0.0167        2796      59.7
Baseline + Lag2                    0.477    0.0164        2767      60.6
Baseline + Cumul                   0.497    0.0162        2579      50.1
Full model (all features)          0.454    0.0128        2789      45.0


In [11]:
from sklearn.linear_model import LinearRegression

ols_predictions = {}

for name, features in feature_sets.items():
    X = df_m[features]
    loo = LeaveOneOut()
    preds = []
    for train_idx, test_idx in loo.split(X):
        m = LinearRegression()
        m.fit(X.iloc[train_idx], y.iloc[train_idx])
        preds.append(m.predict(X.iloc[test_idx])[0])
    ols_predictions[name] = np.array(preds)

print(f"{'Model':<30}{'R2':>8}{'MAE':>10}")
for name, preds in ols_predictions.items():
    r2 = r2_score(y_true, preds)
    mae = mean_absolute_error(y_true, preds)
    print(f"{name:<30}{r2:>8.3f}{mae:>10.0f}")

Model                               R2       MAE
Baseline (N + SST)               0.550      2480
Baseline + ENSO                  0.513      2618
Baseline + Lag2                  0.592      2402
Baseline + Cumul                 0.532      2567
Full model (all features)        0.527      2571


In [12]:
ols_baseline_abs_err = np.abs(y_true - ols_predictions['Baseline (N + SST)'])

for name in feature_sets:
    if name == 'Baseline (N + SST)':
        continue
    alt_abs_err = np.abs(y_true - ols_predictions[name])
    diff = ols_baseline_abs_err - alt_abs_err
    stat, p = wilcoxon(diff)
    print(f"{name:<30} W={stat:.1f}  p={p:.4f}")

Baseline + ENSO                W=254.0  p=0.0927
Baseline + Lag2                W=349.0  p=0.7634
Baseline + Cumul               W=264.0  p=0.1252
Full model (all features)      W=348.0  p=0.7524


In [13]:
from sklearn.inspection import permutation_importance

full_features = feature_sets['Full model (all features)']
X_full = df_m[full_features]

# Fit one RF on all data (not LOO) so permutation_importance has a model to work with
m_full = RandomForestRegressor(n_estimators=100, random_state=42)
m_full.fit(X_full, y)

result = permutation_importance(
    m_full, X_full, y,
    n_repeats=30, random_state=42, scoring='r2'
)

print(f"{'Feature':<20}{'Importance':>12}{'Std':>10}")
for i in result.importances_mean.argsort()[::-1]:
    print(f"{full_features[i]:<20}{result.importances_mean[i]:>12.4f}{result.importances_std[i]:>10.4f}")

Feature               Importance       Std
nitrogen_load             1.4113    0.2036
nitrogen_cumul2           0.1493    0.0420
sst_c                     0.0684    0.0139
oni_summer                0.0482    0.0089
nitrogen_lag2             0.0287    0.0061
oni_spring                0.0224    0.0046
enso_x_nitrogen           0.0129    0.0032
